# Demo workshop using sample dataset nyctaxi

Catalog uses a three-level namespace: **catalog.schema.table**

In [0]:
%sql
SELECT COUNT(*) AS n_trips,
       MIN(tpep_pickup_datetime) AS first_pickup,
       MAX(tpep_pickup_datetime) AS last_pickup,
       ROUND(AVG(fare_amount), 2) AS avg_fare
FROM samples.nyctaxi.trips;

**Visualization** build-in

In [0]:
%sql
SELECT hour(tpep_pickup_datetime) AS pickup_hour,
       COUNT(*) AS trips,
       ROUND(AVG(fare_amount), 2) AS avg_fare,
       ROUND(AVG(trip_distance), 2) AS avg_distance
FROM samples.nyctaxi.trips
GROUP BY 1
ORDER BY 1;

**Databricks Assistant**
Example phrase : Show the 10 most common pickup zip codes with their average fare

Data samples are *read only*, you can make your own, stored here on databricks

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workshop_demo;

pyspark build-in databricks. Used to stream and clean data  
Making a silver table

In [0]:
from pyspark.sql import functions as F

trips = spark.table('samples.nyctaxi.trips')

clean = (
    trips
    .withColumn('trip_minutes',
        (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 60)
    .withColumn('pickup_hour', F.hour('tpep_pickup_datetime'))
    .filter((F.col('trip_distance') > 0)
            & (F.col('fare_amount') > 0)
            & (F.col('trip_minutes') > 0)
            & (F.col('trip_minutes') < 240))
)

print('raw rows  :', trips.count())
print('clean rows:', clean.count())

clean.write.mode('overwrite').saveAsTable('workshop_demo.trips_clean')

Gold level summary table

In [0]:
zip_stats = (
    spark.table('workshop_demo.trips_clean')
    .groupBy('pickup_zip')
    .agg(F.count('*').alias('trips'),
         F.round(F.avg('fare_amount'), 2).alias('avg_fare'),
         F.round(F.avg('trip_distance'), 2).alias('avg_distance'),
         F.round(F.avg('trip_minutes'), 1).alias('avg_minutes'))
    .orderBy(F.desc('trips'))
)
zip_stats.write.mode('overwrite').saveAsTable('workshop_demo.zip_stats')
display(zip_stats.limit(10))

**Delta Lake** superpowers  
ACID transactions, versioning, time travel and rollback on plain files in cloud storage.

In [0]:
%sql
DESCRIBE HISTORY workshop_demo.trips_clean;

In [0]:
%sql
DELETE FROM workshop_demo.trips_clean WHERE trip_distance > 25;

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM workshop_demo.trips_clean VERSION AS OF 0) AS rows_before,
  (SELECT COUNT(*) FROM workshop_demo.trips_clean) AS rows_now;

In [0]:
%sql
RESTORE TABLE workshop_demo.trips_clean TO VERSION AS OF 0;

Automatic lineage capture

A first **ML Model** with mlflow  
With experiments tracking build-in

In [0]:
import mlflow
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

pdf = (spark.table('workshop_demo.trips_clean')
       .select('trip_distance', 'trip_minutes', 'pickup_hour', 'fare_amount')
       .limit(50000).toPandas())

X = pdf[['trip_distance', 'trip_minutes', 'pickup_hour']]
y = pdf['fare_amount']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

mlflow.sklearn.autolog()
with mlflow.start_run(run_name='fare_linear_regression'):
    model = LinearRegression().fit(X_train, y_train)
    mae = mean_absolute_error(y_test, model.predict(X_test))
    mlflow.log_metric('test_mae', mae)
    print('MAE (dollars):', round(mae, 2))